In [0]:
def setup_amfs_environment(spark):
    """Initializes the Catalog and Schemas for the propensity project."""
    
    # 1. Create Catalog
    spark.sql("CREATE CATALOG IF NOT EXISTS amfs_tm")
    
    # 2. Create Schemas
    schemas = ["raw", "clean", "features", "audit"]
    
    for schema in schemas:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS amfs_tm.{schema}")
        print(f"✅ Schema 'amfs_tm.{schema}' is ready.")

setup_amfs_environment(spark)

In [0]:
import os
import re
from pyspark.sql import functions as F

def migrate_to_raw_tables(spark, volume_path, snapshot, catalog="amfs_tm", schema="raw"):
    """
    Ingests all listed Bank Mandiri text files into Delta tables with a _raw suffix.
    """
    # Define the specific files to be migrated
    files_to_migrate = [
        "axa_avr", "axa_cust_cd", "axa_cust_demo", "axa_tran_allper_cd",
        "axa_tran_allper_db", "axa_tran_net", "cc_type_cnt", "cif_acct_mapping",
        "cifsumm", "cust_zipcode_mapping", "dc_type_cnt", "debitcard_trx_edc_cif",
        "master_filter", "master_filter_consent", "phsumm", "qris_trx_edc_cif",
        "trx_outflow", "trx_outflow_payment_purchase"
    ]

    for file_base in files_to_migrate:
        # Construct file path: /path/to/vol/202505/axa_avr_202505.txt
        file_path = f"{volume_path}/{snapshot}/{file_base}_{snapshot}.txt"
        target_table = f"{catalog}.{schema}.bm_{file_base}_raw"
        
        if not os.path.exists(file_path.replace("/dbfs", "")):
            print(f"⚠️ Warning: File not found, skipping {file_path}")
            continue

        print(f"🚀 Migrating: {file_base} -> {target_table}")
        
        try:
            # BM files are typically pipe-delimited
            df = spark.read.option("header", "true").option("sep", "|").csv(file_path)
            
            # Add snapshot_date column for partitioning
            df = df.withColumn("snapshot_date", F.lit(snapshot))
            
            # Write as Delta Table
            df.write.format("delta").mode("append") \
                .partitionBy("snapshot_date") \
                .option("overwriteSchema", "true") \
                .saveAsTable(target_table)
            
            print(f"✅ Success: {target_table} updated.")
            
        except Exception as e:
            print(f"❌ Error migrating {file_base}: {str(e)}")

#Execution
migrate_to_raw_tables(spark, "/Volumes/workspace/default/amfs_tm/data_from_BM", "202505")

In [0]:
import os
from pyspark.sql import functions as F

def migrate_amfs_to_raw(spark, volume_path, snapshot, catalog="amfs_tm", schema="raw"):
    """
    Ingests AMFS interaction and status files into Delta tables with _raw suffix.
    """
    # Mapping filename to target table and its specific separator
    amfs_files = {
        "ALL_STATUS": {"table": "all_status", "sep": ","},
        "TSO": {"table": "tso", "sep": "\t"},
        "call_tracking_savings": {"table": "call_tracking", "sep": ","}
    }

    for file_key, meta in amfs_files.items():
        ext = ".csv" if meta["sep"] == "," else ".txt"
        file_path = f"{volume_path}/{snapshot}/{file_key}_{snapshot}{ext}"
        target_table = f"{catalog}.{schema}.amfs_{meta['table']}_raw"
        
        if not os.path.exists(file_path.replace("/dbfs", "")):
            print(f"⚠️ Warning: File not found, skipping {file_path}")
            continue

        print(f"🚀 Migrating AMFS: {file_key} -> {target_table}")
        
        try:
            df = spark.read.option("header", "true").option("sep", meta["sep"]).csv(file_path)
            
            # Add snapshot_date for partitioning
            df = df.withColumn("snapshot_date", F.lit(snapshot))
            
            df.write.format("delta").mode("append") \
                .partitionBy("snapshot_date") \
                .option("overwriteSchema", "true") \
                .saveAsTable(target_table)
            
            print(f"✅ Success: {target_table} updated.")
        except Exception as e:
            print(f"❌ Error migrating {file_key}: {str(e)}")

# Execution:
migrate_amfs_to_raw(spark, "/Volumes/workspace/default/amfs_tm/data_from_AMFS", "202505")